# Fine-tune Flan-T5-base on SAMSum

Week 1 run for `benchmarks/summarization_results.md` (PROJECT_PLAN.md §5, Lahari).

**Before running:** Settings → Accelerator **GPU T4 x2** (or P100), Internet **on**; Add-ons → Secrets → `HF_TOKEN` (a Hugging Face *write* token) attached to this notebook. Use *Save Version → Save & Run All* so the run survives closing the tab (~2–3 h).

If the session dies, set `RESUME = True` and run again: training continues from `last-checkpoint/` on the Hub.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# which code to train with: `dev` once week 1 is merged, the weekly branch before that
BRANCH = "week1-lahari-samsum-finetune"
RUN = "flan-t5-base-samsum"
RESUME = False

!rm -rf repo && git clone --depth 1 --branch $BRANCH https://github.com/Mounika-Reddy-0802/AI-Meeting-Summarizer-NLP.git repo
!cd repo && git log -1 --format='%h %s'

In [ ]:
# Kaggle already has a CUDA torch; install everything else
!grep -v '^torch' repo/ml/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
HUB_MODEL_ID = f"{whoami()['name']}/{RUN}"
print("checkpoints go to", HUB_MODEL_ID)

In [ ]:
!cd repo && python ml/data/prepare.py --dataset samsum

In [ ]:
resume = "--resume_from_checkpoint hub" if RESUME else ""
!cd repo && python ml/train.py --dataset samsum --epochs 3 --run_name $RUN \n    --push_to_hub --hub_model_id $HUB_MODEL_ID $resume

## Test-set evaluation

Scores zero-shot `google/flan-t5-base` and our best checkpoint on all 819 SAMSum test dialogues (ROUGE + BERTScore, beam 4) on the same GPU, and writes both rows to `benchmarks/raw/results.csv`. The extractive row is already in the csv from a laptop run (it needs no GPU).

In [ ]:
!cd repo && python ml/evaluate.py --system zero-shot --model google/flan-t5-base --batch_size 32

In [ ]:
!cd repo && python ml/evaluate.py --system finetuned --model ml/checkpoints/$RUN/final \n    --label $HUB_MODEL_ID --batch_size 32

## Outputs to commit

Download `/kaggle/working/outputs` from the *Output* tab and copy the files into the repo: `results.csv` and the predictions files into `benchmarks/raw/`, `run_info.json` into `benchmarks/raw/train_flan-t5-base-samsum.json`. Then run `python ml/evaluate.py --table` locally, add the notebook version link and run time to the weekly doc, and commit.

In [ ]:
!mkdir -p outputs && cp repo/benchmarks/raw/results.csv outputs/ \n  && cp repo/benchmarks/raw/predictions/samsum_test_*.jsonl outputs/ \n  && cp repo/ml/checkpoints/$RUN/final/run_info.json outputs/ && ls -la outputs